**Deep Learning Final Project - Algorithmic Bias in Image Processing: Ethical Implications and Mitigation Strategies**

CNN Models: Ethical implications in gender classification, CNN Model- **ConvNeXt_base Model**

**Author: Gomathi Ramesh (GomRam)**

   - COMP 647 (Deep Learning)
   - Rice University

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import json
from collections import Counter


In [ ]:

!ls /kaggle/input


In [ ]:

WEIGHTS_PATH = "/kaggle/input/convnext-base-1k-224-ema-pth/convnext_base_1k_224_ema.pth"

import os
print("weights exist?", os.path.exists(WEIGHTS_PATH))



In [ ]:
# Set up

class Config:
    """Configuration for FairFace preprocessing"""
    DATA_DIR = '/kaggle/input/fairface/fairface'
    TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')
    VAL_IMG_DIR = os.path.join(DATA_DIR, 'val')
    TRAIN_LABELS_CSV = os.path.join(DATA_DIR, 'fairface_label_train.csv')
    VAL_LABELS_CSV = os.path.join(DATA_DIR, 'fairface_label_val.csv')
    
    OUTPUT_DIR = '/kaggle/working/processed_data'
    
    # Split ratios (combining train/val for new split)
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # Image preprocessing
    IMG_SIZE = 224
    NORMALIZE_MEAN = [0.485, 0.456, 0.406]
    NORMALIZE_STD = [0.229, 0.224, 0.225]
    
    # Random seed for reproducibility
    RANDOM_SEED = 42
    
    # Batch size for DataLoader
    BATCH_SIZE = 32
    NUM_WORKERS = 2

# Create output directory
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

In [ ]:
# load data then parse and encode labels

def load_and_parse_labels():
    """
    Load and parse FairFace labels from CSV files using Config paths.
    
    Returns:
        pd.DataFrame: Combined dataframe with columns:
            - file: image filename
            - age: age group (kept for integrity)
            - gender: target variable (Male/Female)
            - race: protected attribute (7 classes)
            - split_source: original split (train/val)
    """
    print("Loading label files...")
    print(f"Train CSV: {Config.TRAIN_LABELS_CSV}")
    print(f"Val CSV: {Config.VAL_LABELS_CSV}")
    
    # Load CSVs directly from Config
    train_df = pd.read_csv(Config.TRAIN_LABELS_CSV)
    val_df = pd.read_csv(Config.VAL_LABELS_CSV)
    
    # Add source split identifier
    train_df['split_source'] = 'train'
    val_df['split_source'] = 'val'
    
    # Combine datasets
    df = pd.concat([train_df, val_df], ignore_index=True)
    
    print(f"Total images loaded: {len(df)}")
    print(f"\nDataset columns: {df.columns.tolist()}")
    
    # Display dataset stats
    print("\n" + "="*60)
    print("DATASET STATISTICS")
    print("="*60)
    
    print(f"\nGender distribution:")
    print(df['gender'].value_counts())
    
    print(f"\nRace distribution:")
    print(df['race'].value_counts())
    
    print(f"\nAge distribution:")
    print(df['age'].value_counts())
    
    return df

def encode_labels(df):
    """
    Encode categorical labels to numerical values.
    
    Returns:
        df: DataFrame with added encoded columns
        label_encodings: Dict with encoding mappings
    """
    print("\nEncoding labels...")
    
    # Gender encoding (binary classification)
    gender_map = {'Male': 0, 'Female': 1}
    df['gender_encoded'] = df['gender'].map(gender_map)
    
    # Race encoding (protected attribute)
    race_categories = df['race'].unique()
    race_map = {race: idx for idx, race in enumerate(sorted(race_categories))}
    df['race_encoded'] = df['race'].map(race_map)
    
    label_encodings = {
        'gender': gender_map,
        'race': race_map
    }
    
    print(f"Gender encoding: {gender_map}")
    print(f"Race encoding: {race_map}")
    
    # Save encodings
    with open(os.path.join(Config.OUTPUT_DIR, 'label_encodings.json'), 'w') as f:
        json.dump(label_encodings, f, indent=2)
    
    return df, label_encodings


In [ ]:
# splits

def create_stratified_splits(df):
    """
    Create stratified train/val/test splits maintaining race and gender distribution.
    
    Returns:
        train_df, val_df, test_df: Split dataframes
    """
    print("\n" + "="*60)
    print("CREATING STRATIFIED SPLITS")
    print("="*60)
    
    # Create stratification col combining gender and race
    df['stratify_col'] = df['gender'].astype(str) + '_' + df['race'].astype(str)
    
    # First split: separate test set
    train_val_df, test_df = train_test_split(
        df,
        test_size=Config.TEST_RATIO,
        stratify=df['stratify_col'],
        random_state=Config.RANDOM_SEED
    )
    
    # Second split: separate train and val
    relative_val_size = Config.VAL_RATIO / (Config.TRAIN_RATIO + Config.VAL_RATIO)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        stratify=train_val_df['stratify_col'],
        random_state=Config.RANDOM_SEED
    )
    
    print(f"\nSplit sizes:")
    print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
    
    # Verify
    for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        print(f"\n{split_name} set distribution:")
        print(f"  Gender: {dict(split_df['gender'].value_counts())}")
        print(f"  Race counts: {len(split_df['race'].value_counts())} categories")
    
    # Save split indices
    split_info = {
        'train_indices': train_df.index.tolist(),
        'val_indices': val_df.index.tolist(),
        'test_indices': test_df.index.tolist(),
        'random_seed': Config.RANDOM_SEED
    }
    
    with open(os.path.join(Config.OUTPUT_DIR, 'split_indices.json'), 'w') as f:
        json.dump(split_info, f)
    
    print(f"\nSplit indices saved to: {Config.OUTPUT_DIR}/split_indices.json")
    
    return train_df, val_df, test_df

In [ ]:
# resizing, cropping, adjust color/brightness/contrast/normalizing/etc
# and creating data loaders

class FairFaceDataset(Dataset):
    """
    PyTorch Dataset for FairFace images with gender classification and race attributes.
    """
    def __init__(self, dataframe, img_dirs, transform=None):
        """
        Args:
            dataframe: DataFrame with image paths and labels
            img_dirs: Dict mapping split_source to image directories
            transform: torchvision transforms to apply
        """
        self.df = dataframe.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Construct image path
        # The 'file' column contains paths like 'train/12345.jpg' or 'val/67890.jpg'
        # need to extract just the filename and use split_source to get the right directory
        filename = os.path.basename(row['file'])  # Extract just the filename (like '12345.jpg')
        split_source = row['split_source']
        img_path = os.path.join(self.img_dirs[split_source], filename)
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image as fallback
            image = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE))
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Prepare labels
        gender_label = row['gender_encoded']
        race_label = row['race_encoded']
        
        return {
            'image': image,
            'gender': torch.tensor(gender_label, dtype=torch.long),
            'race': torch.tensor(race_label, dtype=torch.long),
            'filename': row['file']
        }

def get_transforms(split='train'):
    """
    Get appropriate transforms for train/val/test splits.
    
    Args:
        split: 'train', 'val', or 'test'
    
    Returns:
        torchvision.transforms.Compose
    """
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((Config.IMG_SIZE + 32, Config.IMG_SIZE + 32)),
            transforms.RandomCrop(Config.IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=Config.NORMALIZE_MEAN, std=Config.NORMALIZE_STD)
        ])
    else:  # val or test
        return transforms.Compose([
            transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
            transforms.CenterCrop(Config.IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=Config.NORMALIZE_MEAN, std=Config.NORMALIZE_STD)
        ])

def create_dataloaders(train_df, val_df, test_df):
    """
    Create PyTorch DataLoaders for all splits.
    
    Returns:
        train_loader, val_loader, test_loader
    """
    print("\n" + "="*60)
    print("CREATING DATALOADERS")
    print("="*60)
    
    # Image directory mapping
    img_dirs = {
        'train': Config.TRAIN_IMG_DIR,
        'val': Config.VAL_IMG_DIR
    }
    
    # Create datasets
    train_dataset = FairFaceDataset(
        train_df, 
        img_dirs, 
        transform=get_transforms('train')
    )
    
    val_dataset = FairFaceDataset(
        val_df, 
        img_dirs, 
        transform=get_transforms('val')
    )
    
    test_dataset = FairFaceDataset(
        test_df, 
        img_dirs, 
        transform=get_transforms('test')
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )
    
    print(f"\nDataLoader created:")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches:   {len(val_loader)}")
    print(f"  Test batches:  {len(test_loader)}")
    
    return train_loader, val_loader, test_loader

In [ ]:
# test and baseline fairness stats

def verify_dataloaders(train_loader, val_loader, test_loader):
    """
    Verify dataloaders work correctly and check for data leakage.
    """
    print("\n" + "="*60)
    print("VERIFYING DATALOADERS")
    print("="*60)
    
    # Test loading a batch
    print("\nTesting batch loading...")
    batch = next(iter(train_loader))
    
    print(f"Batch keys: {batch.keys()}")
    print(f"Image batch shape: {batch['image'].shape}")
    print(f"Gender labels shape: {batch['gender'].shape}")
    print(f"Race labels shape: {batch['race'].shape}")
    print(f"Gender labels (first 10): {batch['gender'][:10].tolist()}")
    print(f"Race labels (first 10): {batch['race'][:10].tolist()}")
    
    # Check image value range
    img_min = batch['image'].min().item()
    img_max = batch['image'].max().item()
    print(f"\nImage value range: [{img_min:.3f}, {img_max:.3f}]")
    
    print("\n DataLoaders verified successfully!")

def compute_fairness_metrics_baseline(train_df, val_df, test_df):
    """
    Compute baseline fairness statistics before model training.
    """
    print("\n" + "="*60)
    print("BASELINE FAIRNESS STATISTICS")
    print("="*60)
    
    for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        print(f"\n{split_name} Set:")
        
        # Gender distribution by race
        print("\nGender distribution by race:")
        crosstab = pd.crosstab(split_df['race'], split_df['gender'], normalize='index') * 100
        print(crosstab.round(2))
        
        # Check for imbalances
        for race in split_df['race'].unique():
            race_subset = split_df[split_df['race'] == race]
            gender_dist = race_subset['gender'].value_counts(normalize=True)
            imbalance_ratio = gender_dist.max() / gender_dist.min()
            print(f"  {race}: imbalance ratio = {imbalance_ratio:.2f}")

In [ ]:
# main

def main():
    """
    Main execution pipeline for FairFace data preprocessing.
    """
    print("="*60)
    print("FAIRFACE DATA PREPROCESSING PIPELINE")
    print("="*60)
    
    # Set random seeds for reproducibility
    np.random.seed(Config.RANDOM_SEED)
    torch.manual_seed(Config.RANDOM_SEED)
    
    # Step 1: Load and parse labels
    df = load_and_parse_labels()
    
    # Step 2: Encode labels
    df, label_encodings = encode_labels(df)
    
    # Step 3: Create stratified splits
    train_df, val_df, test_df = create_stratified_splits(df)
    
    # Step 4: Save split DataFrames
    train_df.to_csv(os.path.join(Config.OUTPUT_DIR, 'train_split.csv'), index=False)
    val_df.to_csv(os.path.join(Config.OUTPUT_DIR, 'val_split.csv'), index=False)
    test_df.to_csv(os.path.join(Config.OUTPUT_DIR, 'test_split.csv'), index=False)
    print(f"\nSplit CSV files saved to: {Config.OUTPUT_DIR}")
    
    # Step 5: Create DataLoaders
    train_loader, val_loader, test_loader = create_dataloaders(train_df, val_df, test_df)
    
    # Step 6: Verify DataLoaders
    verify_dataloaders(train_loader, val_loader, test_loader)
    
    # Step 7: Compute baseline fairness metrics
    compute_fairness_metrics_baseline(train_df, val_df, test_df)
    
    print("\n" + "="*60)
    print("PREPROCESSING COMPLETE!")
    print("="*60)
    print(f"\nOutputs saved to: {Config.OUTPUT_DIR}")
    print("Files created:")
    print("  - label_encodings.json")
    print("  - split_indices.json")
    print("  - train_split.csv")
    print("  - val_split.csv")
    print("  - test_split.csv")
    
    return train_loader, val_loader, test_loader, label_encodings

In [ ]:
if __name__ == "__main__":
    train_loader, val_loader, test_loader, label_encodings = main()

#CNN Models:

**Architecture:**

- Three ImageNet models such as **InceptionResnetV2, EfficientNetV2, and ConvNeXt**
- Load ImageNet-pretrained Model
- Replace final fully-connected layer with 2-unit output for binary gender classification.

Proposed Methods:

Baseline Models: Train ResNet-18 (CNN) and ViT-Base as baseline classifiers, evaluating overall accuracy alongside race stratified metrics including Demographic Parity Gap and Equalized Odds Gap. This establishes whether each architecture reproduces race-dependent disparities.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for batch in tqdm(loader, desc="Training", leave=False):
        images = batch["image"].to(device)
        labels = batch["gender"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return running_loss / total, correct / total


@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_races = [], [], []
    
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        images = batch["image"].to(device)
        labels = batch["gender"].to(device)
        races  = batch["race"].to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_races.append(races.cpu())
    
    return (
        running_loss / total,
        correct / total,
        torch.cat(all_preds),
        torch.cat(all_labels),
        torch.cat(all_races)
    )


In [ ]:
def compute_fairness_metrics(preds, labels, races, num_races=7):
    preds = preds.numpy()
    labels = labels.numpy()
    races = races.numpy()

    per_race = {}
    pos_rate_by_race, tpr_by_race, fpr_by_race = [], [], []

    for r in range(num_races):
        idx = races == r
        if idx.sum() == 0:
            continue

        pr = preds[idx]
        yl = labels[idx]

        pos_rate = pr.mean()
        pos_rate_by_race.append(pos_rate)

        pos_idx = (yl == 1)
        neg_idx = (yl == 0)

        tpr = pr[pos_idx].mean() if pos_idx.sum() > 0 else np.nan
        fpr = pr[neg_idx].mean() if neg_idx.sum() > 0 else np.nan
        tpr_by_race.append(tpr)
        fpr_by_race.append(fpr)

        acc = (pr == yl).mean()

        per_race[r] = {
            "count": int(idx.sum()),
            "accuracy": float(acc),
            "pos_rate": float(pos_rate),
            "tpr": float(tpr) if not np.isnan(tpr) else None,
            "fpr": float(fpr) if not np.isnan(fpr) else None
        }

    dp_gap = float(np.nanmax(pos_rate_by_race) - np.nanmin(pos_rate_by_race)) if len(pos_rate_by_race) else np.nan
    tpr_gap = float(np.nanmax(tpr_by_race) - np.nanmin(tpr_by_race)) if len(tpr_by_race) else np.nan
    fpr_gap = float(np.nanmax(fpr_by_race) - np.nanmin(fpr_by_race)) if len(fpr_by_race) else np.nan
    eo_gap = np.nanmax([tpr_gap, fpr_gap])

    return {
        "per_race": per_race,
        "dp_gap": dp_gap,
        "tpr_gap": tpr_gap,
        "fpr_gap": fpr_gap,
        "eo_gap": float(eo_gap) if not np.isnan(eo_gap) else np.nan
    }


def fairness_report(metrics, race_map=None):
    rows = []
    for r, stats in metrics["per_race"].items():
        race_name = race_map.get(r, str(r)) if race_map else str(r)
        rows.append({
            "race": race_name,
            "count": stats["count"],
            "accuracy": stats["accuracy"],
            "pos_rate": stats["pos_rate"],
            "tpr": stats["tpr"],
            "fpr": stats["fpr"],
        })
    df = pd.DataFrame(rows)
    return df.sort_values("race"), metrics["dp_gap"], metrics["eo_gap"]


In [ ]:
from tqdm import tqdm

WEIGHTS_PATH = "/kaggle/input/convnext-base-1k-224-ema-pth/convnext_base_1k_224_ema.pth"

def build_convnext_baseline(num_classes=2):
    model = timm.create_model("convnext_base", pretrained=False)  
    in_features = model.head.fc.in_features
    model.head.fc = nn.Linear(in_features, num_classes)
    return model

def load_imagenet_weights(model, weights_path=WEIGHTS_PATH):
    state_dict = torch.load(weights_path, map_location="cpu")
    model.load_state_dict(state_dict, strict=False)
    print("ImageNet pretrained ConvNeXt weights loaded.")
    return model

In [ ]:
def run_convnext_baseline(train_loader, val_loader, test_loader, label_encodings,
                          epochs=5, lr=1e-4, weight_decay=1e-4):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_convnext_baseline(num_classes=2)
    model = load_imagenet_weights(model, WEIGHTS_PATH).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0.0
    best_state = None

    for ep in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, _, _, _ = eval_model(model, val_loader, criterion, device)

        print(f"Epoch {ep}/{epochs} | "
              f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
              f"Val loss {val_loss:.4f}, acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc, preds, labels, races = eval_model(model, test_loader, criterion, device)

    metrics = compute_fairness_metrics(preds, labels, races, num_races=len(label_encodings["race"]))
    race_inv_map = {v: k for k, v in label_encodings["race"].items()}
    df_report, dp_gap, eo_gap = fairness_report(metrics, race_map=race_inv_map)

    print("\n=== ConvNeXt Baseline (ImageNet Pretrained) ===")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"DP Gap: {dp_gap:.4f}")
    print(f"EO Gap: {eo_gap:.4f}")
    print(df_report.to_string(index=False))

    return model, df_report, metrics


In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np

cmodel_convnext, df_convnext, metrics_convnext = run_convnext_baseline(
    train_loader, val_loader, test_loader, label_encodings,
    epochs=5, lr=1e-4, weight_decay=1e-4
)


**Evaluation Metrics**

- Compute on the test set:
  - Overall accuracy
  - Race-stratified accuracy
  - Demographic Parity Gap:
  - Equalized Odds Gap:
  - Differences in TPR and FPR across race groups.
- Outputs:
   - Baseline CNN performance + fairness results + confusion matrices + plots.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

test_loss, test_acc, preds, labels, races = eval_model(
    model_convnext, test_loader, criterion, device
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if "torch" in str(type(preds)):
    preds  = preds.cpu().numpy()
    labels = labels.cpu().numpy()
    races  = races.cpu().numpy()

# Race mapping label_encodings
race_inv_map = {v: k for k, v in label_encodings["race"].items()}

# ============================================================
# OVERALL CONFUSION MATRIX
# ============================================================
cm_overall = confusion_matrix(labels, preds, labels=[0,1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_overall,
    display_labels=["Male (0)", "Female (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Overall Confusion Matrix")
plt.show()

print("Overall confusion matrix:\n", cm_overall)


# ============================================================
# PER-RACE CONFUSION MATRICES
# ============================================================
unique_races = sorted(np.unique(races))

for r in unique_races:
    idx = races == r
    if idx.sum() == 0:
        continue

    cm_race = confusion_matrix(labels[idx], preds[idx], labels=[0,1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_race,
        display_labels=["Male (0)", "Female (1)"]
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — Race: {race_inv_map[r]}")
    plt.show()

    print(f"\nRace: {race_inv_map[r]} | Count: {idx.sum()}")
    print(cm_race)


# ============================================================
# PER-RACE ACCURACY BAR PLOT
# ============================================================
acc_list = []
race_names = []

for r in unique_races:
    idx = races == r
    if idx.sum() == 0:
        continue
    acc = (preds[idx] == labels[idx]).mean()
    acc_list.append(acc)
    race_names.append(race_inv_map[r])

plt.figure(figsize=(8,5))
plt.bar(race_names, acc_list)
plt.xticks(rotation=45, ha="right")
plt.ylim(0,1)
plt.title("Per-Race Accuracy")
plt.ylabel("Accuracy")
plt.show()


# ============================================================
# TPR / FPR PER-RACE BAR PLOT  (Equalized Odds components)
# ============================================================
tpr_list = []
fpr_list = []

for r in unique_races:
    idx = races == r
    if idx.sum() == 0:
        continue

    y_true = labels[idx]
    y_pred = preds[idx]

    pos_idx = y_true == 1
    neg_idx = y_true == 0

    tpr = y_pred[pos_idx].mean() if pos_idx.sum() > 0 else np.nan
    fpr = y_pred[neg_idx].mean() if neg_idx.sum() > 0 else np.nan

    tpr_list.append(tpr)
    fpr_list.append(fpr)

x = np.arange(len(race_names))
width = 0.35

plt.figure(figsize=(9,5))
plt.bar(x - width/2, tpr_list, width, label="TPR")
plt.bar(x + width/2, fpr_list, width, label="FPR")
plt.xticks(x, race_names, rotation=45, ha="right")
plt.ylim(0,1)
plt.title("Per-Race TPR / FPR (Equalized Odds Components)")
plt.ylabel("Rate")
plt.legend()
plt.show()


# ============================================================
# POSITIVE PREDICTION RATE PER-RACE (Demographic Parity)
# ============================================================
pos_rate_list = []

for r in unique_races:
    idx = races == r
    if idx.sum() == 0:
        continue
    pos_rate = preds[idx].mean()
    pos_rate_list.append(pos_rate)

plt.figure(figsize=(8,5))
plt.bar(race_names, pos_rate_list)
plt.xticks(rotation=45, ha="right")
plt.ylim(0,1)
plt.title("Per-Race Positive Prediction Rate (Demographic Parity)")
plt.ylabel("P(Ŷ = 1)")
plt.show()
